In [1]:
with open("book_training_data.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()
print('total number of characters: ', len(raw_text))
print(raw_text[:1199])


total number of characters:  456795
Virus programming (basics) #1...
This section is dedicated to those who would like to write a virus, but don't have the
knowledge to do so.  First of all, writing a virus is no big deal.  It is an easy project, but one
which requires some basic programmin g skills, and the desire to write a virus!  If either of
these is missing, writing a virus would be tedious indeed!.
 Well, if you meet these requisites, keep reading this article....
               JE   READ
               JNE  FUCK_YOU!
READ:
The survival of a virus is based in its ability to reproduce.  "So how the fuck do I make a
program reproduce?", you might ask. Simple, by getting it to copy itself to other files....
The functional logic of a virus is as follows:
1- Search for a file to infect
2- Open the file to see if it is infected
3- If infected, search for another file
4- Else, infect the file
5- Return control to the host program.
The following is an example of a simple virus:
;*******

In [2]:
import re
text = raw_text[:99]
result = re.split(r'([,.:;?_!"()\#]|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Virus', 'programming', '(', 'basics', ')', '#', '1', '.', '.', '.', 'This', 'section', 'is', 'dedicated', 'to', 'those', 'who', 'would', 'like', 'to', 'write', 'a', 'virus']


In [3]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['Virus', 'programming', '(', 'basics', ')', '#1', '.', '.', '.', 'This', 'section', 'is', 'dedicated', 'to', 'those', 'who', 'would', 'like', 'to', 'write', 'a', 'virus', ',', 'but', 'don', "'", 't', 'have', 'the', 'knowledge']


In [4]:
print(len(preprocessed))

88214


In [5]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

13095


In [6]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [7]:
for i, item in enumerate(vocab.items()):
  print(item)
  if i >= 5:
    break

('\x1a', 0)
('!', 1)
('"', 2)
('#', 3)
('##', 4)
('############', 5)


In [8]:
class SimpleTokenizerV1:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s, i in vocab.items()}
  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()#\']|--|\s)', text)

    preprocessed = [
        item.strip() for item in preprocessed if item.strip()
    ]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids
  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [9]:
tokenizer = SimpleTokenizerV1(vocab)

text = raw_text[200:300]
ids = tokenizer.encode(text)
print(ids)

[13002, 11269, 111, 7961, 10852, 12896, 11579, 12020, 7761, 11265, 9448, 11976, 111, 7557, 12395, 8654, 12467, 12985, 7344, 12796, 1]


In [10]:
tokenizer.decode(ids)

'y project, but one which requires some basic programmin g skills, and the desire to write a virus!'

In [11]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [12]:
len(vocab.items())

13097

In [13]:
for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)

('ŒLECTRONIC', 13092)
('ŒLectronic', 13093)
('Œzine', 13094)
('<|endoftext|>', 13095)
('<|unk|>', 13096)


In [14]:
class SimpleTokenizerV2:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = { i:s for s, i in vocab.items()}
  def encode(self, text):
    preprocessed = re.split(r'([,.:;?!"()#\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    preprocessed = [
        item if item in self.str_to_int
        else "<|unk|>" for item in preprocessed
    ]
    ids = [self.str_to_int[s] for s in preprocessed]
    return ids
  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.:;?!"()#\'])', r'\1', text)
    return text

In [15]:
tokenizer = SimpleTokenizerV2(vocab)
text1 = "hello, I am a Virus!"
text2 = raw_text[100:200]
text = " <|endoftext|> ".join((text1, text2))

print(text)

hello, I am a Virus! <|endoftext|>  but don't have the
knowledge to do so.  First of all, writing a virus is no big deal.  It is an eas


In [16]:
tokenizer.encode(text)

[9669,
 111,
 4293,
 7535,
 7344,
 6814,
 1,
 13095,
 7961,
 8836,
 44,
 12290,
 9645,
 12395,
 10126,
 12467,
 8818,
 12005,
 244,
 3872,
 10814,
 7500,
 111,
 12989,
 7344,
 12796,
 10017,
 10731,
 7832,
 8560,
 244,
 4466,
 10017,
 7546,
 13096]

In [17]:
tokenizer.decode(tokenizer.encode(text))

"hello, I am a Virus! <|endoftext|> but don' t have the knowledge to do so. First of all, writing a virus is no big deal. It is an <|unk|>"

In [18]:
!pip install tiktoken

In [19]:
import importlib
import tiktoken

print(importlib.metadata.version("tiktoken"))

0.12.0


In [20]:
tokenizer = tiktoken.get_encoding("gpt2")


In [21]:
text = raw_text[100:1000]
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[475, 836, 470, 423, 262, 198, 45066, 284, 466, 523, 13, 220, 3274, 286, 477, 11, 3597, 257, 9471, 318, 645, 1263, 1730, 13, 220, 632, 318, 281, 2562, 1628, 11, 475, 530, 198, 4758, 4433, 617, 4096, 1430, 1084, 308, 4678, 11, 290, 262, 6227, 284, 3551, 257, 9471, 0, 220, 1002, 2035, 286, 198, 27218, 318, 4814, 11, 3597, 257, 9471, 561, 307, 32460, 5600, 43179, 198, 3894, 11, 611, 345, 1826, 777, 1038, 31327, 11, 1394, 3555, 428, 2708, 1106, 198, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 449, 36, 220, 220, 20832, 198, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 449, 12161, 220, 30998, 62, 36981, 0, 198, 15675, 25, 198, 464, 9441, 286, 257, 9471, 318, 1912, 287, 663, 2694, 284, 22919, 13, 220, 366, 2396, 703, 262, 5089, 466, 314, 787, 257, 198, 23065, 22919, 35379, 345, 1244, 1265, 13, 17427, 11, 416, 1972, 340, 284, 4866, 2346, 284, 584, 3696, 1106, 198, 464, 10345, 9156, 286, 257, 9471, 318, 355, 5679, 25, 198, 16, 12, 11140, 329, 2

In [22]:
enc_text = tokenizer.encode(raw_text)

In [23]:
enc_sample = enc_text[50:]
print(enc_sample[100:200])

[198, 15675, 25, 198, 464, 9441, 286, 257, 9471, 318, 1912, 287, 663, 2694, 284, 22919, 13, 220, 366, 2396, 703, 262, 5089, 466, 314, 787, 257, 198, 23065, 22919, 35379, 345, 1244, 1265, 13, 17427, 11, 416, 1972, 340, 284, 4866, 2346, 284, 584, 3696, 1106, 198, 464, 10345, 9156, 286, 257, 9471, 318, 355, 5679, 25, 198, 16, 12, 11140, 329, 257, 2393, 284, 7580, 198, 17, 12, 4946, 262, 2393, 284, 766, 611, 340, 318, 14112, 198, 18, 12, 1002, 14112, 11, 2989, 329, 1194, 2393, 198, 19, 12, 25974, 11, 7580, 262, 2393, 198, 20, 12]


In [24]:
context_size = 6

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:  {y}")

x: [632, 318, 281, 2562, 1628, 11]
y:  [318, 281, 2562, 1628, 11, 475]


In [25]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(context, "-> ", desired)

[632] ->  318
[632, 318] ->  281
[632, 318, 281] ->  2562
[632, 318, 281, 2562] ->  1628
[632, 318, 281, 2562, 1628] ->  11
[632, 318, 281, 2562, 1628, 11] ->  475


In [26]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(tokenizer.decode(context), "-> ", tokenizer.decode([desired]))


 It ->   is
 It is ->   an
 It is an ->   easy
 It is an easy ->   project
 It is an easy project ->  ,
 It is an easy project, ->   but


In [27]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i + max_length]
      target_chunk = token_ids[i+1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [28]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

  dataloader = DataLoader(
      dataset,
      batch_size = batch_size,
      shuffle=shuffle,
      drop_last = drop_last,
      num_workers = num_workers
  )
  return dataloader

In [29]:
import torch
print(torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

2.10.0+cpu
[tensor([[   53, 19397,  8300,   357]]), tensor([[19397,  8300,   357, 12093]])]
[tensor([[19397,  8300,   357, 12093]]), tensor([[ 8300,   357, 12093,   873]])]


In [30]:
input_ids = torch.tensor([2, 3, 5, 1])

In [31]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [32]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [33]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [34]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [35]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)


In [38]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [39]:
print(inputs)
print(inputs.shape)

tensor([[   53, 19397,  8300,   357],
        [12093,   873,     8,  1303],
        [   16,   986,   198,  1212],
        [ 2665,   318,  7256,   284],
        [  883,   508,   561,   588],
        [  284,  3551,   257,  9471],
        [   11,   475,   836,   470],
        [  423,   262,   198, 45066]])
torch.Size([8, 4])


In [40]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [41]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [43]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [46]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings)

tensor([[[-1.7494,  0.0854, -0.5665,  ...,  0.0391, -0.3994, -0.0438],
         [ 0.9954, -0.8336,  0.0532,  ..., -1.5936,  0.5158,  2.2580],
         [ 2.3623, -2.3645, -0.5706,  ...,  1.1998,  0.0755,  0.9936],
         [ 0.5147,  0.4693,  1.3349,  ...,  0.4043,  1.2873,  2.2402]],

        [[-1.6899, -2.2702, -0.2164,  ...,  2.0797, -1.5001,  1.0671],
         [ 0.7487,  0.2661,  0.7132,  ..., -0.7997, -1.5964,  1.6900],
         [-0.5420, -0.5485,  1.3976,  ...,  1.1865, -0.0149,  0.9517],
         [ 2.1345,  0.8768,  0.6839,  ..., -0.7812,  0.2894, -0.1195]],

        [[ 0.0113,  0.4856, -2.3064,  ...,  0.2712, -0.3535,  1.0998],
         [ 1.7331,  0.4413, -0.3157,  ..., -0.1841, -1.0161,  0.1800],
         [ 0.4369, -2.3628,  1.9183,  ...,  1.1648, -0.6076,  0.3041],
         [ 1.8723,  1.4082,  0.9492,  ...,  0.9082,  1.1784,  0.1349]],

        ...,

        [[-1.5876,  0.2158,  1.2213,  ...,  2.9723, -2.4064, -0.0771],
         [-0.0390,  0.0336,  0.0293,  ...,  0.0913, -0.00